# LightGBM Volatility Backtest

CPU-only tree-based walk-forward backtest. LightGBM handles NaN natively,
so no scaling or missing-value imputation is needed for features.

In [ ]:
# export
"""LightGBM volatility backtest executor."""

import argparse
import json
import os

import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor

from src.evaluation import apply_duan_smearing
from src.loading import apply_overnight_fills, load_raw_data, parse_exog_cols
from src.scaling import run_backtest
from src.transforms import (
    PERIODS_PER_DAY,
    add_calendar_features,
    apply_horizon_shift,
    generate_har_features,
    resolve_har_lags,
    robust_transform,
)


In [ ]:
# â”€â”€ Setup: clone repo, install deps â”€â”€
import os

REPO_URL = "https://github.com/jamesdchen/harxhar.git"
REPO_DIR = "/content/harxhar"
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
!pip install -q numpy pandas scikit-learn pyarrow tqdm lightgbm

import sys  # noqa: E402

sys.path.insert(0, "/content/harxhar/colab")

In [ ]:
# â”€â”€ Parameters â”€â”€
HORIZON = 1
TRAIN_WINDOW_DAYS = 500
PERIODS_PER_DAY = 48
REFIT_FREQUENCY = 5
DATA_PATH = "all30min"

In [ ]:
# â”€â”€ Load + transform â”€â”€
import numpy as np

from src.loading import load_raw_data
from src.transforms import robust_transform

df = load_raw_data(DATA_PATH, allow_missing=True)
print(f"Loaded: {df.shape}")

# Full transform on RV target (diurnal + winsor)
adj_rv, baseline = robust_transform(df, "RV", is_target=True, use_diurnal=True, winsor_window=240)
df["adj_RV"] = adj_rv
df["baseline"] = baseline

print(f"adj_RV stats:\n{df['adj_RV'].describe()}")

In [ ]:
# --- Features from shared pipeline ---
from src.transforms import resolve_har_lags, generate_har_features, add_calendar_features

df, har_names = generate_har_features(df, target_col="adj_RV")
cal_names = add_calendar_features(df)
feature_names = har_names + cal_names

print(f"HAR lags: {resolve_har_lags()}")
print(f"Features ({len(feature_names)}): {feature_names}")

max_lag = resolve_har_lags()[-1]
df = df.iloc[max_lag:].reset_index(drop=True)
print(f"Shape after lag trim: {df.shape}")

In [ ]:
# --- LightGBM walk-forward backtest ---
from lightgbm import LGBMRegressor
from src.transforms import apply_horizon_shift
from src.scaling import run_backtest

X = df[feature_names].values.astype(np.float64)
y = df["adj_RV"].values.astype(np.float64)
dates = df["t"]
baselines = df["baseline"].values.astype(np.float64)

X, y, dates, baselines = apply_horizon_shift(X, y, dates, baselines, HORIZON)

train_win = TRAIN_WINDOW_DAYS * PERIODS_PER_DAY
print(f"Samples: {len(y)}, Features: {X.shape[1]}, Train window: {train_win}")

model_fn = lambda: LGBMRegressor(
    n_estimators=500, max_depth=5, learning_rate=0.1,
    n_jobs=-1, verbose=-1,
)

predictions = run_backtest(model_fn, X, y, train_win=train_win, refit_frequency=REFIT_FREQUENCY, use_scaling=False)
print(f"Predictions: {predictions.shape}, NaN count: {np.isnan(predictions).sum()}")

# Refit final model for feature importance
final_model = model_fn()
final_model.fit(X[-train_win:], y[-train_win:])
model = final_model

In [ ]:
# â”€â”€ Feature importance â”€â”€
import matplotlib.pyplot as plt

importance = model.feature_importances_
sorted_idx = np.argsort(importance)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh([feature_names[i] for i in sorted_idx], importance[sorted_idx])
ax.set_xlabel("Feature Importance (split count)")
ax.set_title("LightGBM Feature Importance")
plt.tight_layout()
plt.show()

In [ ]:
# --- Evaluation ---
from src.evaluation import apply_duan_smearing, calculate_metrics
import pandas as pd

oos_start = train_win
y_oos = y[oos_start:]
dates_oos = dates.iloc[oos_start:].values
baselines_oos = baselines[oos_start:]

pred_raw, true_raw = apply_duan_smearing(predictions, y_oos, baselines_oos)

results_df = pd.DataFrame({
    "date": dates_oos, "horizon": HORIZON,
    "true_adj": y_oos, "pred_adj": predictions,
    "true_raw": true_raw, "pred_raw": pred_raw,
})

metrics = calculate_metrics(results_df)
print("\n".join(f"{k:>10s}: {v:.6f}" for k, v in metrics.items()))

In [ ]:
# export
def main() -> None:
    parser = argparse.ArgumentParser(description="LightGBM walk-forward backtest")
    parser.add_argument("--data-path", default="all30min")
    parser.add_argument("--horizon", type=int, default=1)
    parser.add_argument("--train-window", type=int, default=500, help="training window in days")
    parser.add_argument(
        "--refit-frequency",
        type=int,
        default=1,
        help="how often to refit the model during walk-forward (1 = every step)",
    )
    parser.add_argument("--start", type=int, default=0)
    parser.add_argument("--end", type=int, default=-1)
    parser.add_argument("--exog-cols", default=None, help="Pipe-separated exog columns, e.g. vix|sentiment")
    parser.add_argument("--output-file", required=True)
    parser.add_argument("--params-file", default=None, help="JSON file with tuned hyperparams")
    args = parser.parse_args()

    tuned_params = {}
    if args.params_file:
        with open(args.params_file) as f:
            tuned_params = json.load(f)

    exog_cols = parse_exog_cols(args.exog_cols)

    train_win_periods = args.train_window * PERIODS_PER_DAY

    df = load_raw_data(args.data_path, allow_missing=True)
    if exog_cols:
        apply_overnight_fills(df, exog_cols)
        df = df.dropna(subset=["RV"]).reset_index(drop=True)

    adj_rv, baseline = robust_transform(df, "RV", is_target=True, use_diurnal=True, winsor_window=240)
    df["adj_RV"] = adj_rv
    df["baseline"] = baseline

    adj_exog_cols: list[str] = []
    for col in exog_cols:
        adj_col = f"adj_{col}"
        adj_series, _ = robust_transform(df, col, use_transform=True, use_diurnal=True)
        df[adj_col] = adj_series
        adj_exog_cols.append(adj_col)

    df, har_names = generate_har_features(df, target_col="adj_RV", exog_cols=adj_exog_cols)
    cal_names = add_calendar_features(df)
    feature_names = har_names + cal_names

    max_lag = resolve_har_lags()[-1]
    df = df.iloc[max_lag:].reset_index(drop=True)

    X = df[feature_names].values.astype(np.float64)
    y = df["adj_RV"].values.astype(np.float64)
    dates = df["t"]
    baselines = df["baseline"].values.astype(np.float64)

    X, y, dates, baselines = apply_horizon_shift(X, y, dates, baselines, args.horizon)

    start = args.start
    end = len(X) if args.end == -1 else args.end
    X_chunk, y_chunk = X[start:end], y[start:end]
    dates_chunk = dates.iloc[start:end].reset_index(drop=True)
    baselines_chunk = baselines[start:end]

    if train_win_periods >= len(X_chunk):
        raise ValueError(f"train_window ({train_win_periods}) >= chunk size ({len(X_chunk)})")

    def model_fn():
        defaults = dict(
            n_estimators=500,
            max_depth=5,
            learning_rate=0.1,
            n_jobs=-1,
            verbose=-1,
        )
        defaults.update(tuned_params)
        return LGBMRegressor(**defaults)

    preds = run_backtest(
        model_fn,
        X_chunk,
        y_chunk,
        train_win=train_win_periods,
        refit_frequency=args.refit_frequency,
        use_scaling=False,
    )

    oos_start = train_win_periods
    y_oos = y_chunk[oos_start:]
    dates_oos = dates_chunk.iloc[oos_start:].values
    baselines_oos = baselines_chunk[oos_start:]

    pred_raw, true_raw = apply_duan_smearing(preds, y_oos, baselines_oos)

    results = pd.DataFrame(
        {
            "date": dates_oos,
            "horizon": args.horizon,
            "true_adj": y_oos,
            "pred_adj": preds,
            "true_raw": true_raw,
            "pred_raw": pred_raw,
        }
    )

    os.makedirs(os.path.dirname(args.output_file) or ".", exist_ok=True)
    results.to_csv(args.output_file, index=False)
    from src.evaluation import save_chunk_reduce

    save_chunk_reduce(results, args.output_file)
    print(f"Saved {len(results)} rows -> {args.output_file}")


In [ ]:
# â”€â”€ Smoke-test main() via argv patch (small chunk, no training run) â”€â”€
# Verifies argparse wiring only; actual execution is exercised by the
# interactive cells above plus the executor CLI invoked under HPC.
import sys as _sys

_saved_argv = _sys.argv
try:
    _sys.argv = [
        "ml_lightgbm",
        "--horizon", "1",
        "--train-window", "500",
        "--start", "0",
        "--end", "1",
        "--output-file", "/tmp/ml_lightgbm_smoke.csv",
    ]
    # Only parse args to confirm signature; don't run the full backtest here.
    import argparse as _ap
    _p = _ap.ArgumentParser()
    _p.add_argument("--data-path", default="all30min")
    _p.add_argument("--horizon", type=int, default=1)
    _p.add_argument("--train-window", type=int, default=500)
    _p.add_argument("--start", type=int, default=0)
    _p.add_argument("--end", type=int, default=-1)
    _p.add_argument("--output-file", required=True)
    _p.add_argument("--params-file", default=None)
    _parsed = _p.parse_args(_sys.argv[1:])
    print("main() argparse signature OK:", _parsed)
finally:
    _sys.argv = _saved_argv

In [ ]:
# export
if __name__ == "__main__":
    main()


In [ ]:
# â”€â”€ Regenerate src/ml_lightgbm.py from this notebook â”€â”€
from _exporter import export_notebook
export_notebook("ml_lightgbm.ipynb", "../src/ml_lightgbm.py")